<a href="https://colab.research.google.com/github/lewinskie254/stock-market-prediction/blob/main/predict_stock_prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from tqdm.notebook import tqdm, trange
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [4]:
ticker = "AAPL"
stock = yf.Ticker(ticker)

# Get historical market data (e.g., past 1 year, daily interval)
hist = stock.history(period="1y", interval="1h")
print(hist.head())

# Get basic company info / metadata
info = stock.info
print(info.get("longName"), info.get("regularMarketPrice"))

# Get dividends and stock splits
dividends = stock.dividends
splits = stock.splits

                                 Open        High         Low       Close  \
Datetime                                                                    
2025-10-31 09:30:00-04:00  276.989990  277.320007  269.160004  270.380005   
2025-10-31 10:30:00-04:00  270.429993  271.559906  270.100098  271.299988   
2025-10-31 11:30:00-04:00  271.299896  272.858490  270.220001  272.434998   
2025-10-31 12:30:00-04:00  272.429993  273.170013  270.149994  270.348785   
2025-10-31 13:30:00-04:00  270.329987  272.059906  270.109985  271.809998   

                             Volume  Dividends  Stock Splits  
Datetime                                                      
2025-10-31 09:30:00-04:00  17228178        0.0           0.0  
2025-10-31 10:30:00-04:00   4595066        0.0           0.0  
2025-10-31 11:30:00-04:00   4301242        0.0           0.0  
2025-10-31 12:30:00-04:00   4005527        0.0           0.0  
2025-10-31 13:30:00-04:00   2396660        0.0           0.0  
Apple Inc. 344.63


In [5]:
hist

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2025-10-31 09:30:00-04:00,276.989990,277.320007,269.160004,270.380005,17228178,0.0,0.0
2025-10-31 10:30:00-04:00,270.429993,271.559906,270.100098,271.299988,4595066,0.0,0.0
2025-10-31 11:30:00-04:00,271.299896,272.858490,270.220001,272.434998,4301242,0.0,0.0
2025-10-31 12:30:00-04:00,272.429993,273.170013,270.149994,270.348785,4005527,0.0,0.0
2025-10-31 13:30:00-04:00,270.329987,272.059906,270.109985,271.809998,2396660,0.0,0.0
...,...,...,...,...,...,...,...
2026-10-30 11:30:00-04:00,347.690002,347.880005,345.671112,347.500000,4661066,0.0,0.0
2026-10-30 12:30:00-04:00,347.494995,347.970001,346.244995,346.549988,4717632,0.0,0.0
2026-10-30 13:30:00-04:00,346.540009,346.689911,342.899994,343.134186,8261695,0.0,0.0


In [8]:
hist.isna().sum()

,0
Open,0
High,0
Low,0
Close,0
Volume,0
Dividends,0
Stock Splits,0


In [21]:
def optimizeARIMA(df: pd.DataFrame, order: list):
    results = []

    for o in tqdm(order):
        model = SARIMAX(df, order=o)

        try:
            model_fit = model.fit()
            results.append([model_fit.aic, o])
        except Exception as e:
            print(f"Failed for order {o}: {e}")

    dataframe = pd.DataFrame(results)
    dataframe.columns = ["AIC", "order"]

    return dataframe.sort_values(by="AIC", ascending=True)

In [22]:
from itertools import product

ps = range(0, 5)
ds = range(0, 3)
qs = range(0, 5)

orders = product(ps, ds, qs)
orders = list(orders)

In [23]:
results_df = optimizeARIMA(hist["Close"], orders)

  0%|          | 0/75 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, fr

In [24]:
results_df.head(10)

,AIC,order
69,7039.630892,"(4, 1, 4)"
5,7045.112771,"(0, 1, 0)"
37,7046.140009,"(2, 1, 2)"
20,7047.005681,"(1, 1, 0)"
6,7047.006626,"(0, 1, 1)"
7,7048.969424,"(0, 1, 2)"
35,7048.974528,"(2, 1, 0)"
21,7049.005492,"(1, 1, 1)"
38,7050.115006,"(2, 1, 3)"
8,7050.198273,"(0, 1, 3)"


In [33]:
def trainARIMA(df: pd.DataFrame, order: tuple, horizon: int, train_length: int, window: int)-> pd.DataFrame:
  total_length = train_length + horizon
  results = []
  for i in tqdm(range(train_length, total_length, window)):
    model = SARIMAX(df[:i], order=order)
    try:
      model_fit = model.fit()
      oos_pred = model_fit.forecast(window)
      results.extend(oos_pred)
    except Exception as e:
      print(f"Failed for order {order}: {e}")
  preds_df = pd.DataFrame(results)
  preds_df.columns = ["Predictions"]
  return preds_df


In [34]:
len(hist)

1721

In [35]:
horizon = 720
train_length = len(hist) - horizon
order = (4, 1, 4)
df = hist['Close']
window = 1

In [ ]:
preds = trainARIMA(df, order, horizon, train_length, window)

  0%|          | 0/720 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmod

Failed for order (4, 1, 4): No supported index is available.


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/ml

Failed for order (4, 1, 4): No supported index is available.


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/ml

... (77 outputs left out)

/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/ml

Failed for order (4, 1, 4): No supported index is available.


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/ml

Failed for order (4, 1, 4): No supported index is available.
